# DACON 236749: 제한된 A100 예산으로 비교하는 5개 파이프라인

이 설계는 3개 사전학습 모델의 연산 결과를 공유하는 **추론·실험 의존 그래프**다. GNN을 추가하는 설계가 아니다. 공식 더미 음원은 정답이 없고 세 파일이 동일하므로 입출력 검증에만 쓴다. 성능 비교에는 별도 공개 음원을 사용하며, 그 결과를 비공개 리더보드 성능으로 간주하지 않는다.

```mermaid
flowchart TD
    A[공식 open.zip + 공개 검증 음원·라이선스] --> B[Google Drive 원본 보관]
    B --> C[Colab 로컬 복사·SHA256·라벨/그룹 검사]
    C --> D[원본 16 kHz + 4.0375초 구간]
    D --> P[PANNs: 구간별 음성·음악 존재확률]
    C --> S[HTDemucs: 보컬 / 반주 분리]
    S --> F[DF-Arena: 성분별 구간 fake 확률]
    D --> R[DF-Arena: 분리 전 원본 fake 확률]
    P --> K[파일별 특징 캐시·실행시간·메모리 → Drive]
    F --> K
    R --> K
    K --> G0[G0 공식 베이스라인]
    K --> G1[G1 상위 2개 구간 평균]
    K --> G2[G2 구간별 존재확률 정렬·성분 합집합]
    K --> G3[G3 원본·분리 성분 결합]
    K --> G4[G4 dev 그룹으로 학습한 정규화 로지스틱 결합]
    G0 --> V[고정 holdout: 공식 EER/AUC + 채널별 결과]
    G1 --> V
    G2 --> V
    G3 --> V
    G4 --> V
    V --> Q[성능·시간·메모리·규칙 확인]
    Q --> Z[선택 그래프만 오프라인 ZIP 구성]
    Z --> T[압축 해제 후 실제 script.py 재실행·예측 일치]
    T --> U[DACON 제출·서버 결과 기록]
```

`v_t, m_t`는 구간별 성분 fake 확률, `p_t, q_t`는 구간별 음성·음악 존재확률이다. `r_t`는 분리 전 원본의 fake 확률이다. 모든 집계는 한 파일 안에서만 수행한다.

| 그래프 | 성분 출력과 파일 출력 | 검증할 가설·실패 가능성 |
|---|---|---|
| G0 | V=max(v), M=max(m), P=max(p), Q=max(q), F=max(PV,QM) | 공식 기준점. 분리 왜곡과 한 구간의 오탐에 민감할 수 있다. |
| G1 | V,M을 각각 상위 min(2,K)개 구간 평균으로 변경 | 단발성 오탐을 줄일 수 있지만 짧은 위조 구간을 약화할 수 있다. |
| G2 | V,M,P,Q는 G0. F=max_t[1-(1-p_t v_t)(1-q_t m_t)] | 서로 다른 시간의 존재·위조 증거를 잘못 곱하는 문제를 줄인다. noisy-OR는 독립성 가정이 정확하지 않아 과대평가할 수 있다. |
| G3 | V=.75max(v)+.25max(r), M=.75max(m)+.25max(r), F=max(max(r),PV,QM) | 분리 과정에서 사라진 위조 단서를 보완한다. 원본에서 찾은 위조를 잘못된 성분에 전달할 위험이 있다. |
| G4 | raw/stem 확률 요약·존재확률·불일치·구간 수로 3개 정규화 로지스틱 헤드 학습. 존재 출력은 PANNs 유지 | 연산량 증가 없이 결합을 학습한다. 작은 dev 표본과 데이터 출처의 편향에 과적합할 수 있다. |

G0–G3의 식과 G4의 `C=0.1`은 holdout 평가 전에 고정한다. 단조 스칼라 보정만으로는 EER/AUC의 순위가 바뀌지 않으므로, 이를 별도 개선 모델로 세지 않는다. G4에서 해당 성분의 dev 정답이 부족하면 그 헤드는 G0으로 돌아가고 보고서에 학습 불가를 기록한다.

## 검증과 예산

- 원본·합성 대응본·전화채널 변형·혼합본은 공통 `group_id`로 묶는다. 그룹은 dev와 holdout 양쪽에 나타날 수 없다. 같은 출처의 holdout은 새로운 생성기 전체에 대한 일반화 증거가 아니다.
- 음성에는 노래 보컬도 포함된다. 음악 파일에 보컬이 있는지 확인되지 않으면 음성 정답을 비워 두고 해당 성분 평가에서 제외한다. 음악의 파일 단위 fake 라벨만으로 성분별 fake 라벨을 만들지 않는다.
- 공식 총점은 5개 지표가 모두 계산 가능할 때만 표시한다. 단일 클래스·정답 누락은 평가 불가로 보고한다. 제공 더미에는 성능 점수를 계산하지 않는다.
- A100 연결부터 최대 120분. 실행 코드는 파일 경계에서 시간을 확인하고 중간 결과를 Drive에 저장한다. 종료·패키지 검증 시간을 남겨 특징 추출은 기본 100분 한도로 시작한다.
- 원본 다운로드·Drive 업로드·코드 준비는 GPU 연결 전에 한다. 1B 전체 미세조정과 새 대형 모델 추가는 이번 비교에 포함하지 않는다. 실제 실행시간을 확인한 뒤 표본 수를 조절한다.
- 제출 서버는 L4 22.4 GiB, 1,200개 파일에 60분이다. A100 측정만으로 L4 시간 준수를 주장하지 않는다. 개별 파일과 60초 입력의 시간·메모리를 기록하고 최종 서버 결과를 확인한다.
- 하루 3회 제한이 있으므로 5개 모두를 즉시 제출하지 않는다. 기존 공식 베이스라인 1회를 기준으로 검증을 통과한 후보를 선택한다.

대회 조건·원문 출처는 [COMPETITION_CONTRACT.md](https://drive.google.com/file/d/1-LAwzjSsw6a61x2axDcvfdY4spmPIzct/view), 외부 음원의 권리와 선정은 [DATA_SOURCES.md](https://drive.google.com/file/d/1dG3zSqvw_zQZXcs8lrkdzc0JhmocHVFN/view)에 기록한다. 이 문서는 실험 전 설계이며 성능 향상 결과를 주장하지 않는다.


## 1. Drive 연결과 2시간 예산
A100을 선택한 뒤 순서대로 실행합니다. 데이터·결과는 `MyDrive/DACON_236749`에 저장합니다. 종료 시 런타임을 해제합니다.

In [ ]:
import time, json, shutil, subprocess, sys, zipfile, hashlib, threading
from pathlib import Path
SESSION_START = globals().get('SESSION_START', time.time())
from google.colab import drive, runtime
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive/DACON_236749')
WORK = Path('/content/dacon236749')
WORK.mkdir(exist_ok=True)
RUN = DRIVE / 'runs' / time.strftime('%Y%m%d_%H%M%S', time.gmtime(SESSION_START))
RUN.mkdir(parents=True, exist_ok=True)
HARD_DEADLINE = SESSION_START + 7200
(RUN / 'budget.json').write_text(json.dumps({'start_unix': SESSION_START, 'hard_deadline': HARD_DEADLINE, 'max_gpu_seconds': 7200}))
def budget_stop():
    try:
        (RUN / 'budget_stop.json').write_text(json.dumps({'time': time.time(), 'reason': 'two_hour_limit'}))
    finally:
        runtime.unassign()
watchdog = threading.Timer(max(1, HARD_DEADLINE - time.time() - 30), budget_stop)
watchdog.daemon = True
watchdog.start()
print('Drive:', DRIVE, '\nResults:', RUN, '\nBudget: <=120 minutes')

## 2. 실행 환경과 Drive 데이터 준비
기존 Colab의 torch/torchaudio를 사용하고 필요한 추론 라이브러리를 설치합니다. 실제 설치 버전과 GPU는 결과에 기록됩니다.

In [ ]:
packages = ['demucs==4.0.1', 'panns-inference==0.1.1', 'transformers==4.57.6',
            'librosa==0.10.2.post1', 'diffq==0.2.4', 'einops==0.8.1', 'scikit-learn==1.8.0', 'soundfile']
with (RUN / 'install.log').open('w') as log:
    result = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages],
                            stdout=log, stderr=subprocess.STDOUT, timeout=600)
print((RUN / 'install.log').read_text()[-6000:])
result.check_returncode()
for name in ('open.zip', 'validation.zip'):
    if not (DRIVE / name).is_file():
        raise FileNotFoundError(f'Upload {name} to {DRIVE} before running')
    shutil.copy2(DRIVE / name, WORK / name)
with zipfile.ZipFile(WORK / 'open.zip') as archive:
    archive.extractall(WORK / 'official')
original = next((WORK / 'official').rglob('baseline_submit.zip'))
with original.open('rb') as stream:
    actual_sha = hashlib.file_digest(stream, 'sha256').hexdigest()
assert actual_sha == 'cc86f143915c95fd958a0066cbf14b0db9f4c41fd213310c3650b9fe870d0c70', 'Official ZIP differs'
BASELINE = WORK / 'baseline'
with zipfile.ZipFile(original) as archive:
    archive.extractall(BASELINE)
public_data = next(p for p in (WORK / 'official').rglob('sample_submission.csv')).parent
shutil.copytree(public_data, BASELINE / 'data', dirs_exist_ok=True)
with zipfile.ZipFile(WORK / 'validation.zip') as archive:
    archive.extractall(WORK)
print('Prepared:', BASELINE, WORK / 'validation')

## 3. 재현 가능한 실행 코드
아래 셀은 실제 특징 추출·5개 그래프·공식 지표·패키징 코드입니다.

In [ ]:
%%writefile /content/dacon236749/graph_fusion.py
"""Five file-independent fusion graphs over one shared audio-inference cache."""

from collections.abc import Sequence
from dataclasses import dataclass
from typing import Final, Literal, TypedDict

import numpy as np
from numpy.typing import NDArray


class Feature(TypedDict):
    ID: str
    starts: list[int]
    presence: list[list[float]]
    voice_fake: list[float]
    music_fake: list[float]
    raw_fake: list[float]
    stem_rms: list[float]
    duration: float


class Label(TypedDict):
    ID: str
    group_id: str
    split: Literal["dev", "holdout"]
    FILE_FAKE: int | str | None
    VOICE_FAKE: int | str | None
    MUSIC_FAKE: int | str | None
    VOICE_PRESENT: int | str | None
    MUSIC_PRESENT: int | str | None


class Prediction(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class Head(TypedDict, total=False):
    status: str
    reason: str
    n: int
    train_ids: list[str]
    coef: list[float]
    intercept: float
    mean: list[float]
    scale: list[float]


class Fusion(TypedDict):
    version: int
    feature_names: list[str]
    train_ids: list[str]
    train_groups: list[str]
    heads: dict[str, Head]


@dataclass(frozen=True, slots=True)
class CacheError(ValueError):
    detail: str

    def __str__(self) -> str:
        return self.detail


FEATURE_NAMES: Final = ["voice_max", "voice_mean", "music_max", "music_mean",
    "raw_max", "raw_mean", "voice_presence_max", "music_presence_max",
    "voice_raw_disagreement", "music_raw_disagreement", "log_windows"]
HEADS: Final = ("FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB")
FloatArray = NDArray[np.float64]
GraphPredictions = dict[str, Prediction]


def _arrays(feature: Feature) -> tuple[FloatArray, FloatArray]:
    """Reject corrupted cache probabilities and misaligned window evidence."""
    starts = np.asarray(feature["starts"])
    count = len(starts)
    if not isinstance(feature["ID"], str) or not feature["ID"] or starts.ndim != 1 or count == 0:
        raise CacheError("A nonempty ID and nonempty window starts are required")
    if not np.issubdtype(starts.dtype, np.integer) or np.any(starts < 0) or np.any(np.diff(starts) <= 0):
        raise CacheError("Window starts must be strictly increasing nonnegative integers")
    presence = np.asarray(feature["presence"], dtype=np.float64)
    fake = np.asarray([feature[key] for key in ("voice_fake", "music_fake", "raw_fake")], dtype=np.float64)
    if presence.shape != (count, 2) or fake.shape != (3, count):
        raise CacheError("Presence and fake probabilities must align with every window")
    if any(np.any(~np.isfinite(values) | (values < 0) | (values > 1)) for values in (presence, fake)):
        raise CacheError("Cached probabilities must be finite and inside [0, 1]")
    rms = np.asarray(feature["stem_rms"], dtype=np.float64)
    if rms.shape != (2,) or np.any(~np.isfinite(rms) | (rms < 0)):
        raise CacheError("Stem RMS values must be finite and nonnegative")
    if not np.isfinite(feature["duration"]) or feature["duration"] <= 0:
        raise CacheError("Audio duration must be finite and positive")
    return presence, fake


def _vector(presence: FloatArray, fake: FloatArray) -> FloatArray:
    maxima = fake.max(axis=1)
    return np.concatenate((np.column_stack((maxima, fake.mean(axis=1))).ravel(),
        presence.max(axis=0), np.abs(maxima[:2] - maxima[2]), [np.log1p(fake.shape[1])]))


def feature_vector(feature: Feature) -> FloatArray:
    """Return the fixed, file-local 11-feature input used by the learned heads."""
    return _vector(*_arrays(feature))


def _row(identifier: str, values: Sequence[float]) -> Prediction:
    file_fake, voice_fake, music_fake, voice_present, music_present = values
    return {"ID": identifier, "FILE_FAKE_PROB": round(float(file_fake), 10),
        "VOICE_FAKE_PROB": round(float(voice_fake), 10),
        "MUSIC_FAKE_PROB": round(float(music_fake), 10),
        "VOICE_PRESENT_PROB": round(float(voice_present), 10),
        "MUSIC_PRESENT_PROB": round(float(music_present), 10)}


def graph_predictions(feature: Feature, fusion: Fusion | None = None) -> GraphPredictions:
    """Apply all five frozen graphs; G4 falls back per head until fitted."""
    presence, fake = _arrays(feature)
    vp, mp = presence.max(axis=0)
    vf, mf, raw = fake.max(axis=1)
    baseline = _row(feature["ID"], [max(vf * vp, mf * mp), vf, mf, vp, mp])
    top_vf, top_mf = np.sort(fake[:2], axis=1)[:, -min(2, fake.shape[1]):].mean(axis=1)
    aligned = 1.0 - (1.0 - presence[:, 0] * fake[0]) * (1.0 - presence[:, 1] * fake[1])
    hybrid_vf, hybrid_mf = 0.75 * fake[:2].max(axis=1) + 0.25 * raw
    rows = {"G0": baseline,
        "G1": _row(feature["ID"], [max(top_vf * vp, top_mf * mp), top_vf, top_mf, vp, mp]),
        "G2": _row(feature["ID"], [aligned.max(), vf, mf, vp, mp]),
        "G3": _row(feature["ID"], [max(raw, hybrid_vf * vp, hybrid_mf * mp), hybrid_vf, hybrid_mf, vp, mp]),
        "G4": baseline.copy()}
    if fusion is not None:
        if fusion["version"] != 1 or fusion["feature_names"] != FEATURE_NAMES:
            raise CacheError("Fusion model version or feature order is incompatible")
        vector = _vector(presence, fake)
        for name in HEADS:
            head = fusion["heads"][name]
            if head["status"] == "trained":
                coef, mean, scale = (np.asarray(head[key], dtype=np.float64) for key in ("coef", "mean", "scale"))
                if any(a.shape != vector.shape or np.any(~np.isfinite(a)) for a in (coef, mean, scale)) or np.any(scale <= 0) or not np.isfinite(head["intercept"]):
                    raise CacheError("Fusion parameters must be finite with positive scales")
                logit = float(((vector - mean) / scale) @ coef + head["intercept"])
                probability = np.exp(-np.logaddexp(0.0, -logit))
                rows["G4"][name] = round(float(probability), 10)
            elif head["status"] != "fallback":
                raise CacheError("Unknown fusion head status")
    return rows


def _binary(value: int | str | None) -> int | None:
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        raise CacheError("Ground-truth labels must be binary or missing")
    return int(value)


def fit_fusion(features: Sequence[Feature], labels: Sequence[Label]) -> Fusion:
    """Fit C=0.1 logistic heads on eligible dev rows, never held-out sources."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler

    by_id = {f["ID"]: f for f in features}
    if len(by_id) != len(features) or len({row["ID"] for row in labels}) != len(labels):
        raise CacheError("Feature and label IDs must each be unique")
    groups: dict[str, set[str]] = {"dev": set(), "holdout": set()}
    for row in labels:
        if row["split"] not in groups or not isinstance(row["group_id"], str) or not row["group_id"]:
            raise CacheError("Labels require a dev/holdout split and nonempty group_id")
        source_groups = row["group_id"].split("|")
        if not all(source_groups):
            raise CacheError("Mixtures require nonempty constituent source groups")
        groups[row["split"]].update(source_groups)
        for key in ("FILE_FAKE", "VOICE_FAKE", "MUSIC_FAKE", "VOICE_PRESENT", "MUSIC_PRESENT"):
            _binary(row.get(key))
    if groups["dev"] & groups["holdout"]:
        raise CacheError("Source group overlap between dev and holdout")
    development = sorted((row for row in labels if row["split"] == "dev" and row["ID"] in by_id), key=lambda row: row["ID"])
    vectors = {row["ID"]: feature_vector(by_id[row["ID"]]) for row in development}
    result: Fusion = {"version": 1, "feature_names": FEATURE_NAMES.copy(),
        "train_ids": [row["ID"] for row in development],
        "train_groups": sorted({group for row in development for group in row["group_id"].split("|")}), "heads": {}}
    for name, mask in zip(HEADS, (None, "VOICE_PRESENT", "MUSIC_PRESENT")):
        target = name.removesuffix("_PROB")
        eligible = [row for row in development if _binary(row.get(target)) is not None and (mask is None or _binary(row.get(mask)) == 1)]
        y = np.asarray([_binary(row[target]) for row in eligible], dtype=np.int64)
        head: Head = {"status": "fallback", "reason": "fewer_than_two_classes",
            "n": len(eligible), "train_ids": [row["ID"] for row in eligible]}
        if np.unique(y).size == 2:
            x = np.stack([vectors[row["ID"]] for row in eligible])
            scaler = StandardScaler().fit(x)
            model = LogisticRegression(C=0.1, max_iter=1000, random_state=42).fit(scaler.transform(x), y)
            if np.any(model.n_iter_ >= model.max_iter):
                raise CacheError(f"Fusion head {name} failed to converge")
            head.update(status="trained", reason="", coef=model.coef_[0].tolist(),
                intercept=float(model.intercept_[0]), mean=scaler.mean_.tolist(), scale=scaler.scale_.tolist())
        result["heads"][name] = head
    return result


In [ ]:
%%writefile /content/dacon236749/graph_scoring.py
"""Submission validation and DACON 236749's discrete ROC scoring rule.

Labels are 0/1/None under the official prediction names without ``_PROB``.
Partial submetrics expose coverage; an official total requires complete labels.
"""

from collections.abc import Mapping, Sequence
from itertools import groupby
from math import isfinite
from typing import Final, TypedDict

PREDICTION_COLUMNS: Final = (
    "FILE_FAKE_PROB", "VOICE_FAKE_PROB", "MUSIC_FAKE_PROB",
    "VOICE_PRESENT_PROB", "MUSIC_PRESENT_PROB",
)
SUBMISSION_COLUMNS: Final = ("ID", *PREDICTION_COLUMNS)
LABEL_COLUMNS: Final = tuple(name.removesuffix("_PROB") for name in PREDICTION_COLUMNS)
Row = Mapping[str, str | float | int | None]


class PredictionRow(TypedDict):
    ID: str
    FILE_FAKE_PROB: float
    VOICE_FAKE_PROB: float
    MUSIC_FAKE_PROB: float
    VOICE_PRESENT_PROB: float
    MUSIC_PRESENT_PROB: float


class MetricsReport(TypedDict):
    score: float | None
    metrics: dict[str, float | None]
    invalid: dict[str, str]
    partial: dict[str, str]
    counts: dict[str, int]
    unknown_counts: dict[str, int]
    coverage: dict[str, float]


def _identifier(row: Row) -> str:
    identifier = row.get("ID")
    if not isinstance(identifier, str) or not identifier or identifier != identifier.strip():
        message = f"ID must be a non-empty, unpadded string: {identifier!r}"
        raise ValueError(message)
    return identifier


def validate_predictions(rows: Sequence[Row], expected_ids: Sequence[str]) -> list[PredictionRow]:
    """Parse probabilities and enforce exact columns, IDs, and submission order."""
    if not rows or len(set(expected_ids)) != len(expected_ids):
        message = "Predictions must be non-empty and expected IDs must be unique"
        raise ValueError(message)
    identifiers = [_identifier(row) for row in rows]
    if len(set(identifiers)) != len(identifiers) or identifiers != list(expected_ids):
        message = "Prediction IDs must match the expected unique IDs in order"
        raise ValueError(message)
    parsed = []
    for row, identifier in zip(rows, identifiers, strict=True):
        if set(row) != set(SUBMISSION_COLUMNS):
            message = f"Prediction {identifier} must have exactly the six submission columns"
            raise ValueError(message)
        probabilities = []
        for column in PREDICTION_COLUMNS:
            try:
                probability = float(row[column])
            except (TypeError, ValueError) as error:
                message = f"Invalid probability for {identifier}/{column}"
                raise ValueError(message) from error
            if not isfinite(probability) or not 0 <= probability <= 1:
                message = f"Probability outside finite [0, 1] for {identifier}/{column}"
                raise ValueError(message)
            probabilities.append(probability)
        parsed.append(PredictionRow(
            ID=identifier, FILE_FAKE_PROB=probabilities[0], VOICE_FAKE_PROB=probabilities[1],
            MUSIC_FAKE_PROB=probabilities[2], VOICE_PRESENT_PROB=probabilities[3],
            MUSIC_PRESENT_PROB=probabilities[4],
        ))
    return parsed


def _label(row: Row, column: str) -> int | None:
    value = row.get(column)
    if value is None or value == "":
        return None
    if value not in (0, 1, "0", "1"):
        message = f"Label must be 0, 1, or unknown for {_identifier(row)}/{column}"
        raise ValueError(message)
    return int(value)


def _roc_metrics(targets: Sequence[int], scores: Sequence[float]) -> tuple[float, float]:
    """Match sklearn's full tied-score ROC and the published nearest-point EER."""
    positives = sum(targets)
    negatives = len(targets) - positives
    points = [(0.0, 0.0)]
    true_positives = false_positives = 0
    ranked = sorted(zip(scores, targets, strict=True), reverse=True)
    for _, tied in groupby(ranked, key=lambda pair: pair[0]):
        for _, target in tied:
            true_positives += target
            false_positives += 1 - target
        points.append((false_positives / negatives, true_positives / positives))
    fpr, tpr = min(points, key=lambda point: abs(point[0] - (1 - point[1])))
    fnr = 1 - tpr
    eer = (fpr + fnr) / 2
    auc = sum(
        (right[0] - left[0]) * (right[1] + left[1]) / 2
        for left, right in zip(points, points[1:])
    )
    return eer, auc


def official_metrics(labels: Sequence[Row], predictions: Sequence[Row]) -> MetricsReport:
    """Report known-subset metrics, but require complete truth for an official total.

    Coverage is known eligible labels / (known eligible + unknown relevant labels).
    Unknown presence is relevant because it leaves component eligibility unknown.
    """
    parsed = validate_predictions(predictions, [_identifier(row) for row in labels])
    truth = [{column: _label(row, column) for column in LABEL_COLUMNS} for row in labels]
    report = MetricsReport(
        score=None, metrics={}, invalid={}, partial={}, counts={}, unknown_counts={}, coverage={},
    )
    specifications = (
        ("file_eer", "FILE_FAKE", "FILE_FAKE_PROB", None),
        ("voice_eer", "VOICE_FAKE", "VOICE_FAKE_PROB", "VOICE_PRESENT"),
        ("music_eer", "MUSIC_FAKE", "MUSIC_FAKE_PROB", "MUSIC_PRESENT"),
        ("voice_presence_auc", "VOICE_PRESENT", "VOICE_PRESENT_PROB", None),
        ("music_presence_auc", "MUSIC_PRESENT", "MUSIC_PRESENT_PROB", None),
    )
    for name, target_column, score_column, mask_column in specifications:
        targets = []
        scores = []
        missing = 0
        for row, prediction in zip(truth, parsed, strict=True):
            if mask_column is not None:
                if row[mask_column] is None:
                    missing += 1
                    continue
                if row[mask_column] == 0:
                    continue
            target = row[target_column]
            if target is None:
                missing += 1
                continue
            targets.append(target)
            scores.append(float(prediction[score_column]))
        report["counts"][name] = len(targets)
        report["unknown_counts"][name] = missing
        denominator = len(targets) + missing
        report["coverage"][name] = len(targets) / denominator if denominator else 0.0
        report["metrics"][name] = None
        if missing:
            report["partial"][name] = f"known subset only; unknown relevant labels: {missing}"
        if len(set(targets)) != 2:
            report["invalid"][name] = "both positive and negative labels are required"
            continue
        eer, auc = _roc_metrics(targets, scores)
        report["metrics"][name] = auc if name.endswith("auc") else eer
    if not report["invalid"] and not report["partial"]:
        values = [value for value in report["metrics"].values() if value is not None]
        file_eer, voice_eer, music_eer, voice_auc, music_auc = values
        report["score"] = (
            0.45 * (1 - file_eer) + 0.18 * (1 - voice_eer) + 0.27 * (1 - music_eer)
            + 0.05 * voice_auc + 0.05 * music_auc
        )
    return report


In [ ]:
%%writefile /content/dacon236749/graph_features.py
"""Shared, file-independent model evidence; baseline preprocessing is retained."""

import gc
import hashlib
import importlib.util
import importlib.metadata
import json
import time
from pathlib import Path

import numpy as np
import torch

from graph_fusion import Feature


def atomic_json(path: Path, value) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def digest(path: Path) -> str:
    with path.open('rb') as stream:
        return hashlib.file_digest(stream, 'sha256').hexdigest()


class Extractor:
    """Own the three frozen models and accumulate timing for one experiment."""

    def __init__(self, baseline_script: Path, include_raw: bool = True):
        spec = importlib.util.spec_from_file_location('official_baseline', baseline_script)
        if spec is None or spec.loader is None:
            raise ImportError(f'Cannot import baseline at {baseline_script}')
        self.base = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(self.base)
        self.device = self.base.select_device('cuda')
        self.include_raw = include_raw
        torch.manual_seed(20260913)
        np.random.seed(20260913)
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        self.panns, self.vi, self.mi = self.base.load_panns_model(self.device)
        self.df, self.fake_index = self.base.load_df_arena_model(self.device)
        self.demucs = self.base.load_htdemucs_model().to(self.device)
        self.signature = hashlib.sha256((
            digest(Path(__file__)) + digest(baseline_script)
            + (self.base.MODEL_DIR / 'SHA256SUMS.txt').read_text()
            + '|'.join(importlib.metadata.version(k) for k in
                ('torch', 'torchaudio', 'transformers', 'librosa', 'demucs', 'panns-inference'))
            + f'fp32-serial-shifts0-overlap0.25-raw{include_raw}'
        ).encode()).hexdigest()

    def fake_windows(self, audio: np.ndarray) -> list[float]:
        starts = self.base.get_segment_starts(audio.size)
        if self.base.calculate_rms(audio) < self.base.SILENCE_RMS:
            return [0.0] * len(starts)
        result = []
        for start in starts:
            segment = torch.from_numpy(self.base.extract_segment(audio, start)).to(self.device)
            with torch.inference_mode():
                logits = self.df(input_values=segment)['logits']
                probability = torch.softmax(logits.float(), dim=-1)[0, self.fake_index]
            result.append(float(probability))
        return result

    def extract(self, audio_path: Path) -> Feature:
        """No other file or accumulated prediction enters this computation."""
        audio = self.base.load_audio(audio_path)
        predictions, _ = self.panns.inference(self.base.make_panns_segments(audio))
        presence = np.stack((predictions[:, self.vi].max(1), predictions[:, self.mi].max(1)), 1)
        voice, music = self.base.separate_voice_and_music(audio_path, self.demucs, self.device)
        feature = Feature(
            ID=audio_path.stem, starts=self.base.get_segment_starts(audio.size),
            presence=presence.astype(float).tolist(),
            voice_fake=self.fake_windows(voice), music_fake=self.fake_windows(music),
            raw_fake=self.fake_windows(audio) if self.include_raw else [0.0] * len(presence),
            stem_rms=[self.base.calculate_rms(voice), self.base.calculate_rms(music)],
            duration=float(audio.size / self.base.AUDIO_SAMPLE_RATE),
        )
        lengths = {len(feature[name]) for name in ('starts', 'presence', 'voice_fake', 'music_fake', 'raw_fake')}
        if len(lengths) != 1:
            raise ValueError(f'Window alignment mismatch: {audio_path}, {lengths}')
        return feature

    def close(self) -> None:
        del self.panns, self.df, self.demucs
        gc.collect()
        torch.cuda.empty_cache()


def extract_cached(extractor: Extractor, paths: list[Path], cache_dir: Path, deadline: float) -> list[Feature]:
    """Only complete atomic records can resume; content and code changes invalidate keys."""
    features = []
    for path in paths:
        key = hashlib.sha256((digest(path) + extractor.signature).encode()).hexdigest()
        cache_path = cache_dir / f'{key}.json'
        if cache_path.is_file():
            record = json.loads(cache_path.read_text())
            if record['signature'] != extractor.signature or record['input_sha256'] != digest(path):
                raise ValueError(f'Cache provenance mismatch: {cache_path}')
            feature = record['feature']
        else:
            if time.time() >= deadline:
                break
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()
            start = time.perf_counter()
            feature = extractor.extract(path)
            torch.cuda.synchronize()
            atomic_json(cache_path, {
                'feature': feature, 'input_sha256': digest(path), 'signature': extractor.signature,
                'seconds': time.perf_counter() - start,
                'max_allocated_gib': torch.cuda.max_memory_allocated() / 1024**3,
                'max_reserved_gib': torch.cuda.max_memory_reserved() / 1024**3,
            })
        features.append({**feature, 'ID': path.stem})
        print(f'FEATURE {len(features)}/{len(paths)} {path.stem}', flush=True)
    return features


In [ ]:
%%writefile /content/dacon236749/run_experiment.py
"""A bounded pilot; every GPU result is checkpointed to the mounted Drive."""

import argparse
import csv
import importlib.metadata
import json
import random
import sys
import time
from pathlib import Path

import numpy as np
import soundfile as sf
import torch

from graph_features import Extractor, atomic_json, extract_cached
from graph_fusion import fit_fusion, graph_predictions
from graph_scoring import official_metrics, validate_predictions


def read_csv(path: Path) -> list:
    with path.open(encoding='utf-8-sig', newline='') as stream:
        return list(csv.DictReader(stream))


def write_csv(path: Path, rows: list) -> None:
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def reference_parity(extractor: Extractor, path: Path, feature) -> dict:
    """Compare the shared graph with untouched official functions on real audio."""
    base = extractor.base
    audio = base.load_audio(path)
    vp, mp = base.predict_presence(extractor.panns, extractor.vi, extractor.mi, audio)
    extractor.demucs.cpu()
    voice, music = base.separate_voice_and_music(path, extractor.demucs, extractor.device)
    extractor.demucs.to(extractor.device)
    vf = base.predict_fake(extractor.df, extractor.fake_index, voice, extractor.device)
    mf = base.predict_fake(extractor.df, extractor.fake_index, music, extractor.device)
    expected = np.array([base.combine_file_fake_score(vf, mf, vp, mp), vf, mf, vp, mp])
    actual = np.array(list(graph_predictions(feature)['G0'].values())[1:])
    delta = float(np.max(np.abs(expected - actual)))
    if not np.allclose(expected, actual, rtol=1e-5, atol=1e-6):
        raise AssertionError(f'G0 differs from official inference by {delta}: {path}')
    return {'ID': path.stem, 'max_probability_delta': delta, 'reference': expected.tolist()}


def main() -> None:
    parser = argparse.ArgumentParser()
    parser.add_argument('--baseline', type=Path, required=True)
    parser.add_argument('--validation', type=Path, required=True)
    parser.add_argument('--output', type=Path, required=True)
    parser.add_argument('--deadline', type=float, required=True)
    args = parser.parse_args()
    out = args.output
    out.mkdir(parents=True, exist_ok=True)
    start = time.time()
    if start >= args.deadline:
        raise TimeoutError('The existing GPU budget has expired; no new run was started')
    labels = read_csv(args.validation / 'manifest.csv')
    if len({r['ID'] for r in labels}) != len(labels):
        raise ValueError('Duplicate validation IDs')
    paths = {r['ID']: args.validation / r['path'] for r in labels}
    if any(p.stem != key or not p.is_file() for key, p in paths.items()):
        raise ValueError('Manifest ID/path mismatch or missing audio')
    atomic_json(out / 'environment.json', {
        'python': sys.version, 'gpu': torch.cuda.get_device_name(),
        'gpu_total_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3,
        'started_unix': start, 'deadline_unix': args.deadline,
        'packages': {k: importlib.metadata.version(k) for k in
            ('torch', 'torchaudio', 'transformers', 'numpy', 'scipy', 'scikit-learn', 'demucs', 'librosa')},
    })
    extractor = Extractor(args.baseline / 'script.py')
    try:
        dummy = sorted((args.baseline / 'data' / 'test').glob('*.wav'))
        dummy_features = extract_cached(extractor, dummy, out / 'cache', args.deadline)
        atomic_json(out / 'dummy_features.json', dummy_features)
        ordered = list(paths.values())
        random.Random(20260913).shuffle(ordered)
        first = extract_cached(extractor, ordered[:1], out / 'cache', args.deadline)
        if not first:
            raise TimeoutError('No validation feature completed within the budget')
        atomic_json(out / 'reference_parity.json', reference_parity(extractor, ordered[0], first[0]))
        stress_path = args.baseline / 'stress_60s.wav'
        signal = extractor.base.load_audio(ordered[0])
        sf.write(stress_path, np.tile(signal, 960000 // len(signal) + 1)[:960000], 16000)
        try:
            stress = extract_cached(extractor, [stress_path], out / 'stress_cache', args.deadline)
            atomic_json(out / 'stress60.json', {'completed': bool(stress),
                'purpose': '60-second runtime and memory only; repeated source is not validation evidence'})
        except torch.cuda.OutOfMemoryError as error:
            atomic_json(out / 'stress60.json', {'completed': False, 'error': str(error),
                'submission_supported': False})
            torch.cuda.empty_cache()
        features = extract_cached(extractor, ordered, out / 'cache', args.deadline)
    finally:
        extractor.close()
    atomic_json(out / 'features.json', features)
    available = {f['ID'] for f in features}
    completed_labels = [r for r in labels if r['ID'] in available]
    fusion = fit_fusion(features, labels)
    atomic_json(out / 'fusion.json', fusion)
    predictions = {name: [] for name in ('G0', 'G1', 'G2', 'G3', 'G4')}
    for feature in features:
        for name, row in graph_predictions(feature, fusion).items():
            predictions[name].append(row)
    report = {'complete': len(features) == len(labels), 'completed': len(features),
        'planned': len(labels), 'elapsed_seconds': time.time() - start,
        'scope': 'external_source_group_holdout_not_DACON_leaderboard', 'graphs': {}}
    heldout = [r for r in completed_labels if r['split'] == 'holdout']
    heldout_ids = {r['ID'] for r in heldout}
    heldout_groups = {g for r in heldout for g in r['group_id'].split('|')}
    report['holdout'] = {'clips': len(heldout), 'atomic_source_groups': len(heldout_groups),
        'limitation': 'Transforms and mixtures are correlated; source groups are the independent units'}
    weights = {'file_eer': .45, 'voice_eer': .18, 'music_eer': .27,
        'voice_presence_auc': .05, 'music_presence_auc': .05}
    for name, rows in predictions.items():
        validate_predictions(rows, [f['ID'] for f in features])
        write_csv(out / f'{name}_predictions.csv', rows)
        by_id = {r['ID']: r for r in rows if r['ID'] in heldout_ids}
        selected = [by_id[r['ID']] for r in heldout]
        result = official_metrics(heldout, selected) if heldout else {
            'score': None, 'metrics': {k: None for k in weights},
            'invalid': {k: 'no completed holdout examples' for k in weights},
            'partial': {}, 'counts': {k: 0 for k in weights},
            'unknown_counts': {k: 0 for k in weights}, 'coverage': {k: 0.0 for k in weights}}
        # ponytail: partial-label proxy only selects this pilot; it is never an official score.
        result['selection_proxy'] = sum(weight * ((1 - result['metrics'][metric]) if metric.endswith('eer')
            else result['metrics'][metric]) for metric, weight in weights.items()
            if result['metrics'][metric] is not None)
        report['graphs'][name] = result
        print(name, json.dumps(result, ensure_ascii=False), flush=True)
    ranked = sorted(report['graphs'], key=lambda g: report['graphs'][g]['selection_proxy'], reverse=True)
    best = ranked[0]
    improvement = report['graphs'][best]['selection_proxy'] - report['graphs']['G0']['selection_proxy']
    stress_ok = json.loads((out / 'stress60.json').read_text())['completed']
    eligible = report['complete'] and stress_ok and len(heldout_groups) >= 20 and improvement >= .01
    report['selection'] = {'graph': best if eligible else 'G0', 'best_observed_graph': best,
        'proxy_improvement_over_G0': improvement, 'new_submission_supported': eligible,
        'reason': 'complete_pilot_and_proxy_gain_at_least_0.01' if eligible else 'retain_reference_without_sufficient_evidence'}
    records = [json.loads(p.read_text()) for p in (out / 'cache').glob('*.json')]
    timings = [r['seconds'] for r in records]
    report['runtime'] = {'unique_cached_files': len(records), 'total_extraction_seconds': sum(timings),
        'mean_seconds_per_file': float(np.mean(timings)),
        'max_allocated_gib': max(r['max_allocated_gib'] for r in records),
        'max_reserved_gib': max(r['max_reserved_gib'] for r in records),
        'L4_60_minute_compliance': 'unverified; A100 measurements are not L4 evidence'}
    report['runtime']['stress60'] = [
        {k: v for k, v in json.loads(p.read_text()).items() if k != 'feature'}
        for p in (out / 'stress_cache').glob('*.json')]
    atomic_json(out / 'comparison.json', report)
    for name in predictions:
        dummy_rows = [graph_predictions(f, fusion)[name] for f in dummy_features]
        write_csv(out / f'{name}_dummy_submission.csv', dummy_rows)
    atomic_json(out / 'status.json', {'status': 'comparison_finished', 'unix': time.time(),
        'selected_graph': report['selection']['graph'], 'completed': len(features)})
    print('COMPARISON_FINISHED', json.dumps(report['selection']), flush=True)


if __name__ == '__main__':
    main()


In [ ]:
%%writefile /content/dacon236749/package_submission.py
"""Build a selected graph's offline archive without recompressing model tensors.

The evaluator already supplies the dependencies used here, so requirements.txt
is empty. Colab setup belongs to the experiment notebook, not the submission.
"""

import csv
import json
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path, PurePosixPath
from typing import TYPE_CHECKING, Final, Literal, TypedDict

from graph_scoring import PredictionRow, SUBMISSION_COLUMNS, validate_predictions

if TYPE_CHECKING:
    from graph_fusion import Fusion


class Selection(TypedDict):
    graph: Literal["G0", "G1", "G2", "G3", "G4"]
    fusion: "Fusion | None"


RUNTIME_FILES: Final = ("graph_features.py", "graph_fusion.py", "graph_scoring.py")
MAX_ZIP_BYTES: Final = 10_000_000_000
MAX_EXPANDED_BYTES: Final = 32_000_000_000
DEPLOY_SCRIPT: Final = '''#!/usr/bin/env python3
"""Run the selected frozen graph independently on each evaluation audio file."""
import json
import os
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
RUNTIME = ROOT / "model" / "runtime"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
sys.dont_write_bytecode = True
sys.path.insert(0, str(RUNTIME))

from graph_features import Extractor
from graph_fusion import graph_predictions
from graph_scoring import SUBMISSION_COLUMNS, validate_predictions


def main():
    graph = json.loads((RUNTIME / "choice.json").read_text())["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        raise ValueError("Unsupported graph selection")
    fusion = json.loads((RUNTIME / "fusion.json").read_text())
    extractor = Extractor(RUNTIME / "official_baseline.py", include_raw=graph in {"G3", "G4"})
    try:
        base = extractor.base
        columns, sample = base.read_sample_submission(ROOT / "data" / "sample_submission.csv")
        if columns != list(SUBMISSION_COLUMNS):
            raise ValueError("Sample submission columns differ from the official six-column schema")
        paths = base.order_audio_files(base.find_audio_files(ROOT / "data" / "test"), sample)
        rows = []
        for index, path in enumerate(paths, 1):
            prediction = graph_predictions(extractor.extract(path), fusion)[graph]
            rows.extend(validate_predictions([prediction], [path.stem]))
            print(f"PREDICT {index}/{len(paths)} {path.stem}", flush=True)
        rows = validate_predictions(rows, [row["ID"] for row in sample])
        destination = ROOT / "output" / "submission.csv"
        temporary = destination.with_suffix(".csv.tmp")
        base.save_submission(temporary, columns, rows)
        temporary.replace(destination)
    finally:
        extractor.close()


if __name__ == "__main__":
    main()
'''


def _model_members(archive: zipfile.ZipFile) -> list[zipfile.ZipInfo]:
    members = [member for member in archive.infolist() if member.filename.startswith("model/")]
    names = [member.filename for member in members]
    if not members or len(names) != len(set(names)):
        message = "Baseline model assets must be present with unique archive names"
        raise ValueError(message)
    for member in members:
        path = PurePosixPath(member.filename)
        if path.is_absolute() or ".." in path.parts or "\\" in member.filename:
            message = f"Unsafe baseline member: {member.filename}"
            raise ValueError(message)
        if path.parts[:2] == ("model", "runtime"):
            message = "Baseline already contains reserved model/runtime paths"
            raise ValueError(message)
    return members


def build_submission(original_zip: Path, output_zip: Path, selection: Selection) -> Path:
    """Keep baseline model assets and package one selected graph atomically."""
    if original_zip.resolve() == output_zip.resolve():
        message = "The official baseline archive must not be overwritten"
        raise ValueError(message)
    graph = selection["graph"]
    if graph not in {"G0", "G1", "G2", "G3", "G4"}:
        message = f"Unsupported graph selection: {graph}"
        raise ValueError(message)
    fusion = selection.get("fusion")
    if graph == "G4" and fusion is None:
        message = "G4 packaging requires its frozen fusion parameters"
        raise ValueError(message)
    # G0-G3 cannot accidentally execute unrelated learned-head parameters.
    fusion_json = json.dumps(fusion if graph == "G4" else None, allow_nan=False)
    source_directory = Path(__file__).resolve().parent
    runtime = {f"model/runtime/{name}": (source_directory / name).read_bytes() for name in RUNTIME_FILES}
    runtime["model/runtime/fusion.json"] = fusion_json.encode()
    runtime["model/runtime/choice.json"] = json.dumps({"graph": graph}).encode()
    compile(DEPLOY_SCRIPT, "script.py", "exec")
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    temporary = output_zip.with_name(output_zip.name + ".tmp")
    try:
        with zipfile.ZipFile(original_zip) as source:
            members = _model_members(source)
            baseline = source.read("script.py").decode("utf-8")
            original_base = "BASE_DIR = Path(__file__).resolve().parent"
            if baseline.count(original_base) != 1:
                message = "Official baseline BASE_DIR no longer matches the expected source"
                raise ValueError(message)
            baseline = baseline.replace(original_base, "BASE_DIR = Path(__file__).resolve().parents[2]", 1)
            compile(baseline, "model/runtime/official_baseline.py", "exec")
            runtime["model/runtime/official_baseline.py"] = baseline.encode()
            expanded_bytes = sum(member.file_size for member in members)
            expanded_bytes += sum(map(len, runtime.values())) + len(DEPLOY_SCRIPT.encode())
            if expanded_bytes > MAX_EXPANDED_BYTES or expanded_bytes + 1_000_000 > MAX_ZIP_BYTES:
                message = f"Stored archive would exceed submission limits: {expanded_bytes} bytes"
                raise ValueError(message)
            # Tensor compression costs CPU time without improving the inference result.
            with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED, allowZip64=True) as target:
                for member in members:
                    if member.is_dir():
                        target.writestr(member.filename, b"")
                        continue
                    with source.open(member) as incoming, target.open(member.filename, "w", force_zip64=True) as outgoing:
                        shutil.copyfileobj(incoming, outgoing, length=8 * 1024 * 1024)
                for name, content in runtime.items():
                    target.writestr(name, content)
                target.writestr("script.py", DEPLOY_SCRIPT)
                target.writestr("requirements.txt", b"")
        if temporary.stat().st_size > MAX_ZIP_BYTES:
            message = "Built archive exceeds the compressed submission size limit"
            raise ValueError(message)
        temporary.replace(output_zip)
    finally:
        temporary.unlink(missing_ok=True)
    return output_zip


def verify_packaged(unpacked_dir: Path, deadline: float) -> list[PredictionRow]:
    """Run the real freshly unpacked script and parse its validated predictions.

    The caller provides data/test and data/sample_submission.csv, and compares
    these rows with the notebook's reference predictions. Deadline is Unix time.
    """
    remaining = deadline - time.time()
    if remaining <= 0:
        message = "No experiment time remains for packaged inference"
        raise TimeoutError(message)
    output = unpacked_dir / "output"
    output.mkdir(parents=True, exist_ok=True)
    (output / "submission.csv").unlink(missing_ok=True)
    with (output / "packaged_inference.log").open("w", encoding="utf-8") as log:
        subprocess.run(
            [sys.executable, str((unpacked_dir / "script.py").resolve())],
            cwd=unpacked_dir, stdout=log, stderr=subprocess.STDOUT,
            check=True, timeout=min(remaining, 3600),
        )
    with (unpacked_dir / "data" / "sample_submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        expected_ids = [row["ID"] for row in csv.DictReader(stream)]
    with (output / "submission.csv").open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != list(SUBMISSION_COLUMNS):
            message = "Packaged output must have the exact official six-column header"
            raise ValueError(message)
        rows = list(reader)
    return validate_predictions(rows, expected_ids)


## 4. 실제 A100 비교 실행
특징 추출은 파일마다 Drive에 저장됩니다. 마지막 20분은 패키징·재실행·결과 저장에 남깁니다. 아래 로그의 완료 여부를 확인합니다.

In [ ]:
import os
os.chdir(WORK)
experiment_deadline = min(SESSION_START + 6000, HARD_DEADLINE - 1200)
cmd = [sys.executable, '-u', 'run_experiment.py', '--baseline', str(BASELINE),
       '--validation', str(WORK / 'validation'), '--output', str(RUN),
       '--deadline', str(experiment_deadline)]
with (RUN / 'experiment.log').open('w', buffering=1) as log:
    process = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT)
    while process.poll() is None:
        if time.time() >= HARD_DEADLINE - 900:
            process.terminate()
            process.wait(timeout=30)
            raise TimeoutError('Stopped to preserve packaging and save time')
        time.sleep(15)
        tail = (RUN / 'experiment.log').read_text().splitlines()
        print(tail[-1] if tail else 'Loading models...', flush=True)
if process.returncode:
    print((RUN / 'experiment.log').read_text()[-15000:])
    raise RuntimeError(f'Experiment exited with {process.returncode}; inspect log and resume cached files')
print((RUN / 'experiment.log').read_text()[-12000:])

## 5. 결과 비교와 실제 ZIP 재실행
총점이 `None`이면 성분 정답이나 클래스가 부족한 것입니다. 부분 지표 기반 선택값을 DACON 점수로 해석하지 않습니다.

In [ ]:
import csv, numpy as np
from IPython.display import display, Markdown
from package_submission import build_submission, verify_packaged
comparison = json.loads((RUN / 'comparison.json').read_text())
display(comparison)
selected = comparison['selection']['graph']
fusion = json.loads((RUN / 'fusion.json').read_text())
submission_zip = RUN / f'{selected}_submission.zip'
build_submission(original, submission_zip, {'graph': selected, 'fusion': fusion})
fresh = WORK / 'packaged_check'
with zipfile.ZipFile(submission_zip) as archive:
    archive.extractall(fresh)
shutil.copytree(BASELINE / 'data', fresh / 'data', dirs_exist_ok=True)
rows = verify_packaged(fresh, HARD_DEADLINE - 120)
with (RUN / f'{selected}_dummy_submission.csv').open() as stream:
    reference = list(csv.DictReader(stream))
assert [r['ID'] for r in rows] == [r['ID'] for r in reference]
columns = [k for k in rows[0] if k != 'ID']
difference = max(abs(float(a[k]) - float(b[k])) for a, b in zip(rows, reference) for k in columns)
assert difference <= 1e-5, f'Packaged predictions differ: {difference}'
report = {'graph': selected, 'fresh_offline_script_executed': True,
          'max_probability_delta': difference, 'submission_zip': str(submission_zip),
          'completed_unix': time.time()}
(RUN / 'packaged_verification.json').write_text(json.dumps(report, indent=2))
display(report)

## 6. 저장 확인 후 GPU 해제
Colab 실행 기록은 이 노트북에 저장되며, 원본 특징·예측·지표·로그·선택 ZIP은 위의 Drive 결과 폴더에 남습니다.

In [ ]:
(RUN / 'finished.json').write_text(json.dumps({'status': 'finished', 'elapsed_seconds': time.time() - SESSION_START,
    'comparison': str(RUN / 'comparison.json'), 'note': 'DACON scoring requires a separate server submission'}))
print('FINISHED', RUN, 'GPU elapsed seconds:', round(time.time() - SESSION_START))
runtime.unassign()
watchdog.cancel()